In [ ]:
import json

files = ["test_processed.json", "train_processed.json", "validation_processed.json"]

combined = []

for file in files:
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)
        combined.extend(data)

for i, item in enumerate(combined):
    item["id"] = f"{i:05d}"

with open("data_processed.json", "w", encoding="utf-8") as f:
    json.dump(combined, f, ensure_ascii=False, indent=2)

In [ ]:
import json

files = ["test_tagged.json", "train_tagged.json", "validation_tagged.json"]

combined = []

for file in files:
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)
        combined.extend(data)

for i, item in enumerate(combined):
    item["id"] = f"{i:05d}"

with open("data_tagged.json", "w", encoding="utf-8") as f:
    json.dump(combined, f, ensure_ascii=False, indent=2)

In [ ]:
!pip install stanza -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 28.3 MB/s eta 0:00:00


In [ ]:
import json
import stanza
from tqdm import tqdm

stanza.download("en")
stanza.download("zh")

en_nlp = stanza.Pipeline(
    lang="en",
    processors="tokenize,pos",
    tokenize_pretokenized=True,
    use_gpu=True
)

zh_nlp = stanza.Pipeline(
    lang="zh",
    processors="tokenize,pos",
    tokenize_pretokenized=True,
    use_gpu=True
)


def stanza_tag_tokens(tokens, language_labels):
    pos_tags = []

    for token, lang in zip(tokens, language_labels):
        if lang == "EN":
            doc = en_nlp([[token]])
        else:
            doc = zh_nlp([[token]])

        try:
            upos = doc.sentences[0].words[0].upos
        except Exception:
            upos = "X"

        pos_tags.append([token, upos])

    return pos_tags


input_file = "data_processed.json"
output_file = "data_stanza_gold.json"

with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

new_data = []

for item in tqdm(data):
    tokens = item["tokens"]
    language_labels = item.get("language_labels", item.get("labels"))

    stanza_pos_tags = stanza_tag_tokens(tokens, language_labels)

    new_item = item.copy()
    new_item["gold_pos_tags"] = stanza_pos_tags

    new_data.append(new_item)

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(new_data, f, ensure_ascii=False, indent=2)

print("Saved to:", output_file)

INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading default packages for language: en (English) ...
INFO:stanza:File exists: /root/.cache/stanza/1.11.0/resources/en/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:"zh" is an alias for "zh-hans"
INFO:stanza:Downloading default packages for language: zh-hans (Simplified_Chinese) ...
INFO:stanza:File exists: /root/.cache/stanza/1.11.0/resources/zh-hans/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Loading these models for language: en (English):
| Processor | Package         |
-------------------------------
| tokenize  | combined        |
| pos       | combined_charlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: pos
INFO:stanza:Done loading processors!
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:"zh" is an alias for "zh-hans"
INFO:stanza:Loading these models for language: zh-hans (Simplified_Chinese):
| Processor | Package        |
------------------------------
| tokenize  | gsdsimp        |
| pos       | gsdsimp_charlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: pos
INFO:stanza:Done loading processors!
100%|██████████| 12314/12314 [1:19:48<00:00,  2.57it/s]


Saved to: data_stanza_gold.json
